#### ENRICH `PRODUCTS` FROM BRONZE TO SILVER

In [0]:
products_df = spark.table('ORDERS_SALES.BRONZE.PRODUCTS')
products_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

# Select and cast columns properly
products_enriched_df = (
    products_df
    .select(
        col("product_id"),
        coalesce(initcap(col("product_category_name")), lit("NA")).alias("product_category_name"),
        col("product_name_lenght").cast("float"),
        col("product_description_lenght").cast("float"),
        coalesce(col("product_photos_qty"), lit(0.0)).cast("float").alias("product_photos_qty"),
        col("product_weight_g").cast("float"),
        col("product_length_cm").cast("float"),
        col("product_height_cm").cast("float"),
        col("product_width_cm").cast("float"),
        round(col("product_length_cm").cast("float") * col("product_height_cm").cast("float") * col("product_width_cm").cast("float"), 3).cast("decimal(18,3)").alias("product_volume"),
        col("file_name"),
        col("file_path"),
        current_timestamp().alias("load_timestamp")
    )
    .filter(
        (col("product_id").isNotNull()) &
        (col("product_weight_g") > 0) & (col("product_weight_g") < 10000) &
        (col("product_length_cm") > 0) & (col("product_length_cm") < 200) &
        (col("product_height_cm") > 0) & (col("product_height_cm") < 200) &
        (col("product_width_cm") > 0) & (col("product_width_cm") < 200)
    )
)

# Show one row for verification
products_enriched_df.limit(1).display()

In [0]:
products_enriched_df.createOrReplaceTempView('enriched_products_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED`

In [0]:
%sql
MERGE INTO ORDERS_SALES.SILVER.PRODUCTS_ENRICHED tgt
USING enriched_products_temp_vw src
ON (src.product_id = tgt.product_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.product_id = src.product_id,
    tgt.product_category_name = src.product_category_name,
    tgt.product_name_lenght = src.product_name_lenght,
    tgt.product_description_lenght = src.product_description_lenght,
    tgt.product_photos_qty = src.product_photos_qty,
    tgt.product_weight_g = src.product_weight_g,
    tgt.product_length_cm = src.product_length_cm,
    tgt.product_height_cm = src.product_height_cm,
    tgt.product_width_cm = src.product_width_cm
WHEN NOT MATCHED THEN
INSERT (product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm, product_volume, origin_file_path, load_timestamp)
VALUES (src.product_id, src.product_category_name, src.product_name_lenght, src.product_description_lenght, src.product_photos_qty,src.product_weight_g, src.product_length_cm, src.product_height_cm, src.product_width_cm, src.product_volume, src.file_path, current_timestamp)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.PRODUCTS_ENRICHED')